# Tutorial 2: Local Pools — Filesystem, Redis, HDF5

Store and recall the same entry across three local backends using a single loop.

This demonstrates LAILA's uniform `memorize` / `remember` / `forget` interface — the code is identical regardless of the storage backend.

In [ ]:
import numpy as np
import laila
from laila.pool import FilesystemPool, RedisPool, HDF5Pool

## Create the pools

Each pool manages its own storage automatically:
- **FilesystemPool** — local disk image
- **RedisPool** — private redis-server on a UNIX socket (no system Redis needed)
- **HDF5Pool** — local `.h5py` file

In [ ]:
pools = [
    ("fs",    FilesystemPool(nickname="tutorial_fs")),
    ("redis", RedisPool(nickname="tutorial_redis")),
    ("hdf5",  HDF5Pool(nickname="tutorial_hdf5")),
]

## Create an entry to store

In [ ]:
entry = laila.constant(data=np.random.randn(10, 10), nickname="tutorial_matrix")
print("Entry global_id:", entry.global_id)
print("Data shape:", entry.data.shape)

## Register, memorize, remember, verify — in a loop

The key point: the code inside the loop is identical for all three backends.

In [ ]:
for nick, pool in pools:
    laila.memory.extend(pool, pool_nickname=nick)

    with laila.guarantee:
        laila.memorize(entry, pool_nickname=nick)
    print(f"[{nick}] memorized")

    with laila.guarantee:
        recall_future = laila.remember(entry.global_id, pool_nickname=nick)
    recalled_data = recall_future.data
    print(f"[{nick}] remembered — data shape: {recalled_data.shape}")

    assert np.array_equal(entry.data, recalled_data), f"Data mismatch on {nick}!"
    print(f"[{nick}] verified")

    with laila.guarantee:
        laila.forget(entry.global_id, pool_nickname=nick)
    print(f"[{nick}] cleaned up\n")

## What just happened

1. **`laila.memory.extend`** registered each backend under a nickname.
2. **`laila.memorize`** serialized the entry, applied the pool's transformations, and wrote to the backend.
3. **`with laila.guarantee:`** blocked until every future created inside the block had completed — no manual `wait()` calls needed.
4. **`laila.remember`** fetched the blob, reversed the transformations, and reconstructed the entry. The returned future exposes the recovered payload via `.data`.
5. **`laila.forget`** deleted the stored blob.

Every backend LAILA supports — S3, GCS, Azure, Postgres, MongoDB, and more — answers to the same three verbs. Only the pool constructor changes.

**Next:** Tutorial 3 — S3 with NumPy and PyTorch Tensors